# Diffusion → GenEval

This covers the current baseline experiment from prompts to scores. The official external GenEval evaluator is retained so scores remain comparable.

**Default:** read your completed SD1.5 images/results. For a new run, choose a model, set a fresh `RUN` and `RESULTS`, and enable both switches. Keep the working installations; no installation or download cells are needed.


In [ ]:
import os, json, hashlib, gc, subprocess
from pathlib import Path
import torch, diffusers, pandas as pd
from PIL import Image
from IPython.display import display
from tqdm.auto import tqdm


## 1. Settings
Edit `RESULTS` to the filename that completed successfully. Set `DETECTOR` to the folder containing your successfully redownloaded checkpoint if it differs below. For new runs, use a fresh output folder; there is intentionally no resume/overwrite manager.

`LIMIT=None, SAMPLES=4` gives 553 prompts × 4 images. For smoke testing use `LIMIT=2, SAMPLES=1`. `OFFLOAD=True` trades speed for GPU memory. SD3.5/FLUX require BF16-capable hardware and authorized Hugging Face access; authenticate separately if needed.


In [ ]:
MODEL = "sdxl"  # sd15 | sdxl | sd35 | flux
GENERATE, EVALUATE = True, True
LIMIT, SAMPLES, SEED = None, 4, 67  
OFFLOAD, SAVE_LATENTS = False, False
ROOT = Path("/nfshomes/vyomwal5/DFAR")
GENEVAL = ROOT / "external/geneval"
EVAL_PYTHON = Path("/fs/nexus-scratch/vyomwal5/anaconda3/envs/geneval/bin/python")
DETECTOR = Path(os.environ.get("DETECTOR_DIR", str(GENEVAL / "detector")))
RUN = ROOT / "runs/baseline" / MODEL
RESULTS = ROOT / "results/sdxl_baseline_retry1.jsonl"

# repository, pipeline, resolution, steps, guidance, precision, checkpoint revision
MODELS = {
    "sd15": ("stable-diffusion-v1-5/stable-diffusion-v1-5", "StableDiffusionPipeline", 512, 30, 7.5, "float16", "451f4fe16113bff5a5d2269ed5ad43b0592e9a14"),
    "sdxl": ("stabilityai/stable-diffusion-xl-base-1.0", "StableDiffusionXLPipeline", 1024, 30, 5.0, "float16", "462165984030d82259a11f4367a4eed129e94a7b"),
    "sd35": ("stabilityai/stable-diffusion-3.5-large", "StableDiffusion3Pipeline", 1024, 28, 3.5, "bfloat16", "ceddf0a7fdf2064ea28e2213e3b84e4afa170a0f"),
    "flux": ("black-forest-labs/FLUX.1-dev", "FluxPipeline", 1024, 50, 3.5, "bfloat16", "3de623fc3c33e44ffbe2bad470d0f45bccf2eb21"),
}
print("Images:", RUN, "\nResults:", RESULTS, "\nDetector:", DETECTOR)


## 2. Prompts
Read official benchmark annotations. They describe required objects, counts, colors and relations; there is no unique reference image. The evaluator measures semantic composition, not overall visual quality.


In [ ]:
prompts = [json.loads(x) for x in (GENEVAL / "prompts/evaluation_metadata.jsonl").read_text().splitlines() if x.strip()][:LIMIT]
print(len(prompts), "prompts;", len(prompts) * SAMPLES, "images")
display(pd.DataFrame(prompts)[["tag", "prompt"]].head())


## 3. Generate
One frozen model, one image at a time. The seed formula matches the formal runner, but exact pixels can still depend on software and hardware. Optional native latents are diagnostic; they are not interchangeable across models. Safety filtering remains at the pipeline default.

For a **new** run, enable `GENERATE` and select an unused `RUN`. After generation, set `GENERATE=False` before rerunning the notebook. No training or gradient updates occur.


In [ ]:
def generate():
    repo, cls, size, steps, guidance, dtype, revision = MODELS[MODEL]
    RUN.mkdir(parents=True, exist_ok=False)
    pipe = getattr(diffusers, cls).from_pretrained(repo, revision=revision, torch_dtype=getattr(torch, dtype))
    pipe.enable_model_cpu_offload() if OFFLOAD else pipe.to("cuda")
    pipe.set_progress_bar_config(disable=True)
    latent = {}
    def capture(pipeline, step, timestep, state):
        if step == len(pipeline.scheduler.timesteps) - 1:
            latent["value"] = state["latents"].detach().cpu()
        return state
    callback = {"callback_on_step_end": capture, "callback_on_step_end_tensor_inputs": ["latents"]} if SAVE_LATENTS else {}
    for i, meta in enumerate(tqdm(prompts, desc=MODEL)):
        folder = RUN / f"{i:05d}"
        (folder / "samples").mkdir(parents=True)
        (folder / "metadata.jsonl").write_text(json.dumps(meta) + "\n")
        for j in range(SAMPLES):
            seed = int(hashlib.sha256(json.dumps([SEED, i, j], separators=(",", ":")).encode()).hexdigest()[:8], 16)
            with torch.inference_mode():
                image = pipe(meta["prompt"], height=size, width=size, num_inference_steps=steps,
                             guidance_scale=guidance, generator=torch.Generator("cpu").manual_seed(seed), **callback).images[0]
            image.save(folder / "samples" / f"{j:04d}.png")
            if SAVE_LATENTS:
                torch.save(latent.pop("value"), folder / f"latent_{j:04d}.pt")
    settings = dict(model=MODEL, configuration=MODELS[MODEL], seed=SEED, samples=SAMPLES, limit=LIMIT,
                    offload=OFFLOAD, torch=torch.__version__, diffusers=diffusers.__version__, scheduler=dict(pipe.scheduler.config))
    (RUN / "settings.json").write_text(json.dumps(settings, indent=2))
    del pipe

if GENERATE:
    generate()
gc.collect()
torch.cuda.empty_cache()


## 4. Preview
Inspect appearance separately: a GenEval pass does not guarantee natural shapes or clean details.


In [ ]:
for path in sorted(RUN.glob("*/samples/0000.png"))[:4]:
    print(json.loads((path.parent.parent / "metadata.jsonl").read_text())["prompt"])
    image = Image.open(path)
    image.thumbnail((384, 384))
    display(image)


## 5. Evaluate
Enable `EVALUATE` for a new evaluation and choose an unused results filename. The detector runs under your existing `geneval` Python; `PYTHONNOUSERSITE` avoids the user-site package conflicts you encountered. Keep the same official evaluator checkout and checkpoint used for your baseline.


In [ ]:
if EVALUATE:
    assert not RESULTS.exists(), "Choose a fresh RESULTS filename."
    RESULTS.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run([str(EVAL_PYTHON), str(GENEVAL / "evaluation/evaluate_images.py"), str(RUN),
                    "--outfile", str(RESULTS), "--model-path", str(DETECTOR)],
                   cwd=GENEVAL, env={**os.environ, "PYTHONNOUSERSITE": "1"}, check=True)


## 6. Scores
Check prompt/sample coverage before averaging. The overall GenEval score is the unweighted mean of category accuracies. Smoke subsets are labeled as such. Summary files are written beside the results.

Your completed SD1.5 run should give **macro ≈ 0.422314** and **micro ≈ 0.401447**; these values are a reference, not hard-coded outputs.


In [ ]:
def summarize(path):
    frame = pd.read_json(path, lines=True)
    keys = [(int(Path(f).parent.parent.name), int(Path(f).stem)) for f in frame.filename]
    assert len(keys) == len(set(keys)) == len(prompts) * SAMPLES
    assert set(keys) == {(i, j) for i in range(len(prompts)) for j in range(SAMPLES)}
    for (i, _), row in zip(keys, frame.itertuples()):
        assert row.prompt == prompts[i]["prompt"] and row.tag == prompts[i]["tag"]
    task = frame.groupby("tag").correct.agg(["mean", "count"])
    report = {"images": len(frame), "per_task": task["mean"].to_dict(),
              "overall_macro": float(task["mean"].mean()), "image_micro": float(frame.correct.mean())}
    path.with_suffix(".summary.json").write_text(json.dumps(report, indent=2))
    display(task.rename(columns={"mean": "accuracy", "count": "images"}))
    print("Full baseline" if LIMIT is None and SAMPLES == 4 else "Subset / nonstandard sample count")
    print(json.dumps(report, indent=2))
    return frame

results = summarize(RESULTS)


## 7. Failure examples
Read the evaluator reason alongside the image: the detector can also be wrong. Keep these official evaluation prompts out of correction training. To repeat the baseline for another model, change `MODEL`, `RUN`, `RESULTS`, and the two switches, then run from settings.


In [ ]:
for row in results.loc[~results.correct.astype(bool)].head(4).itertuples():
    print(row.prompt, "\nGenEval:", row.reason)
    path = Path(row.filename)
    image = Image.open(RUN / path.parent.parent.name / "samples" / path.name)
    image.thumbnail((384, 384))
    display(image)


[Official GenEval](https://github.com/djghosh13/geneval) · [Diffusers](https://huggingface.co/docs/diffusers/index)

This is a minimal experiment notebook: no CLI, automatic installs, retry framework, dataset registry, or project-module dependency. Reuse your already working environments. Notebook syntax was checked during preparation; GPU execution must be performed on your server.
